# Restaurant Tips: Data Visualization with Matplotlib & Seaborn

This beginner-friendly notebook explores the Seaborn Tips dataset, checks its quality, answers questions about bills and tips, and builds a final four-chart dashboard. The source values are in US dollars; this project converts bills and tips to Indian rupees using the user-provided rate **1 USD = INR 100**. Run the cells from top to bottom. The reusable plotting functions live in `src/visualization.py`.

## 1. Import libraries and locate the project

Pandas organizes the table, NumPy supports numeric operations, and Matplotlib/Seaborn create the charts. The path setup lets this notebook import the reusable code whether Jupyter is started from the project folder or its `notebooks` folder.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

current = Path.cwd().resolve()
PROJECT_ROOT = current.parent if current.name == "notebooks" else current
sys.path.insert(0, str(PROJECT_ROOT))
from src.visualization import (
    load_and_clean_tips, summarize_data, plot_histogram, plot_average_tips,
    plot_tip_boxplot, plot_bill_tip_scatter, plot_correlation_heatmap,
    create_dashboard,
)

sns.set_theme(style="whitegrid", context="notebook")

## 2. Load and explore the dataset

`load_and_clean_tips()` loads the real dataset with `sns.load_dataset("tips")`. It reports missing values and exact duplicate rows, imputes missing numeric/category values if needed, removes exact duplicates, then converts `total_bill` and `tip` from USD to INR at the selected rate of ₹100 per USD. The original source dataset is not edited.

In [ ]:
tips, cleaning_report = load_and_clean_tips()
print("Cleaning report:", cleaning_report)
print("\nFirst five rows:")
display(tips.head().style.format({"total_bill": "₹{:,.2f}", "tip": "₹{:,.2f}"}))

In [ ]:
print("Dataset information:")
tips.info()

In [ ]:
print("Statistical summary for numeric columns:")
display(tips.describe())
print("Missing values by column:")
display(tips.isna().sum().rename("missing values"))
print("Remaining exact duplicate rows:", tips.duplicated().sum())

## 3. Histogram: total bill distribution

The histogram groups bills into ranges. The y-axis counts tables, while the x-axis reports converted bill amounts in Indian rupees (INR).

In [ ]:
fig = plot_histogram(tips, PROJECT_ROOT / "outputs" / "histogram.png")
plt.show()

## 4. Bar chart: average tips by day

The height of each bar is the mean tip for that day; the whiskers show 95% confidence intervals.

In [ ]:
fig = plot_average_tips(tips, PROJECT_ROOT / "outputs" / "bar_chart.png")
plt.show()

## 5. Box plot: tip distribution by day

Each box summarizes the middle half of observed tips, with the center line marking the median. Points beyond whiskers are potential outliers.

In [ ]:
fig = plot_tip_boxplot(tips, PROJECT_ROOT / "outputs" / "box_plot.png")
plt.show()

## 6. Scatter plot: bill vs tip

Each point represents one table. Lunch and Dinner use different color-blind-friendly colors, making meal-period patterns easy to compare.

In [ ]:
fig = plot_bill_tip_scatter(tips, PROJECT_ROOT / "outputs" / "scatter_plot.png")
plt.show()

## 7. Numerical correlation matrix and heatmap

Pearson correlation ranges from -1 to 1 and describes linear association; it does not prove cause and effect. Converting both monetary values by the same positive factor does not change their correlations. The heatmap annotates each value for easy reading.

In [ ]:
summary = summarize_data(tips)
display(summary["correlations"].round(3))
fig = plot_correlation_heatmap(tips, PROJECT_ROOT / "outputs" / "correlation_heatmap.png")
plt.show()

## 8. Findings from the data

The statements below are calculated from the cleaned dataset in this run. They describe this sample and should not be treated as causal conclusions.

In [ ]:
print(f"Rows analyzed: {summary['rows']}")
print(f"Mean total bill: ₹{summary['mean_total_bill']:,.2f}; median: ₹{summary['median_total_bill']:,.2f}")
print(f"Highest average tip: {summary['highest_average_tip_day']} (₹{summary['highest_average_tip']:,.2f})")
print(f"Total bill-tip Pearson correlation: {summary['bill_tip_correlation']:.3f}")
left, right = summary["strongest_correlation_pair"]
print(f"Strongest numerical correlation: {left} and {right} ({summary['strongest_correlation']:.3f})")
display(summary["daily_mean_tip"].rename("average tip (INR)").to_frame().style.format("₹{:,.2f}"))

## 9. Final 2×2 dashboard

The dashboard combines the bill histogram, tip-by-day box plot, bill-tip scatter plot, and numerical correlation heatmap. Bill and tip axes are in INR using **₹100 per USD**. It is saved as `outputs/tips_overview.png` at 150 DPI.

In [ ]:
dashboard = create_dashboard(tips, PROJECT_ROOT / "outputs" / "tips_overview.png")
plt.show()